# Guardián de inundación comunitario - Análisis con pandas

Este notebook descarga el historial del canal de ThingSpeak, limpia los datos, calcula estadísticas y arma los gráficos.

**Cómo usarlo en Google Colab:** subí este archivo a colab.research.google.com (Archivo > Subir cuaderno) y ejecutá las celdas en orden (Entorno de ejecución > Ejecutar todo).

Mientras no tengas la placa funcionando, dejá `USAR_DATOS_DE_PRUEBA = True`: genera una inundación simulada para que puedas probar todo el análisis.

## 1. Configuración

In [ ]:
CHANNEL_ID = "3533124"        # número de tu canal de ThingSpeak, por ejemplo "1234567"
READ_API_KEY = ""      # solo si el canal es privado (Read API Key)
RESULTADOS = 8000      # ThingSpeak entrega como máximo 8000 registros por pedido
USAR_DATOS_DE_PRUEBA = True   # cambiar a False cuando haya datos reales

UMBRALES_CM = [5, 15, 30, 60]   # aviso, alerta, peligro, evacuación recomendada
ZONA_HORARIA = "America/Argentina/Buenos_Aires"

## 2. Obtener los datos

In [ ]:
import io
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

COLUMNAS = {
    "field1": "nivel_cm",
    "field2": "agua_piso_pct",
    "field3": "velocidad_cm_min",
    "field4": "estado",
    "field5": "respuesta",
    "field6": "dist_cruda_cm",
    "field7": "baseline_cm",
    "field8": "modo_noche",
}

def descargar_thingspeak(channel_id, api_key="", results=8000):
    """Descarga el historial del canal como CSV y lo devuelve como DataFrame."""
    url = f"https://api.thingspeak.com/channels/{channel_id}/feeds.csv"
    params = {"results": results}
    if api_key:
        params["api_key"] = api_key
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return pd.read_csv(io.StringIO(r.text))

def generar_datos_prueba(seed=7):
    """Simula 6 horas con una tormenta: subida del agua, sensor del piso y ruido de sensor."""
    rng = np.random.default_rng(seed)
    n = 6 * 3 * 60                      # una lectura cada 20 s durante 6 horas
    t = pd.date_range("2026-11-05 21:00", periods=n, freq="20s", tz="UTC")
    x = np.arange(n)
    nivel = np.clip(70 * np.exp(-((x - 600) ** 2) / (2 * 160 ** 2)) + rng.normal(0, 0.6, n), 0, None)
    agua_piso = np.clip(np.where(nivel > 3, 70 + nivel, nivel * 15) + rng.normal(0, 4, n), 0, 100)
    vel = np.gradient(nivel) * 3                                          # cm por minuto (lecturas cada 20 s)
    estado = np.digitize(nivel, UMBRALES_CM)
    nivel[rng.choice(n, 6, replace=False)] += rng.uniform(40, 90, 6)      # lecturas falsas
    nivel[rng.choice(n, 12, replace=False)] = np.nan                      # huecos
    respuesta = np.where(estado >= 3, 1, 0)
    df = pd.DataFrame({
        "created_at": t.strftime("%Y-%m-%d %H:%M:%S UTC"),
        "entry_id": np.arange(1, n + 1),
        "field1": nivel, "field2": agua_piso.round(0), "field3": vel.round(1),
        "field4": estado, "field5": respuesta,
        "field6": (150 - nivel).round(1), "field7": 150.0, "field8": 1,
    })
    return df

if USAR_DATOS_DE_PRUEBA:
    crudo = generar_datos_prueba()
    print("Usando datos de prueba simulados")
else:
    crudo = descargar_thingspeak(CHANNEL_ID, READ_API_KEY, RESULTADOS)
    print("Descargados desde ThingSpeak")

print(crudo.shape)
crudo.head()

## 3. Limpieza

Pasos: renombrar columnas, convertir la fecha a hora de Argentina, pasar todo a número, sacar duplicados y valores imposibles, y detectar lecturas falsas (picos aislados) con una mediana móvil.

In [ ]:
df = crudo.rename(columns=COLUMNAS).copy()

# Fecha y hora local
df["created_at"] = pd.to_datetime(df["created_at"], utc=True).dt.tz_convert(ZONA_HORARIA)

# Todo a número (lo que no se pueda convertir queda como NaN)
cols_num = list(COLUMNAS.values())
df[cols_num] = df[cols_num].apply(pd.to_numeric, errors="coerce")

antes = len(df)
df = df.drop_duplicates(subset="created_at").sort_values("created_at").reset_index(drop=True)
print("Duplicados eliminados:", antes - len(df))

print("Nulos por columna:")
print(df[cols_num].isna().sum())

# Rangos válidos
df.loc[~df["nivel_cm"].between(0, 150), "nivel_cm"] = np.nan
df.loc[~df["agua_piso_pct"].between(0, 100), "agua_piso_pct"] = np.nan

# Lecturas falsas: pico aislado que se aleja mucho de la mediana móvil
mediana = df["nivel_cm"].rolling(5, center=True, min_periods=1).median()
df["es_atipico"] = (df["nivel_cm"] - mediana).abs() > 15
print("Valores atípicos detectados:", int(df["es_atipico"].sum()))

df["nivel_limpio"] = df["nivel_cm"].mask(df["es_atipico"])
df["nivel_limpio"] = df["nivel_limpio"].interpolate(limit=6)   # rellena huecos cortos
df = df.dropna(subset=["nivel_limpio"]).reset_index(drop=True)
df.head()

## 4. Estadística descriptiva

In [ ]:
print(df[["nivel_limpio", "agua_piso_pct", "velocidad_cm_min"]].describe().round(2))

# Tiempo en cada estado (suma de los intervalos entre lecturas)
paso = df["created_at"].diff().dt.total_seconds().fillna(0).clip(upper=120)
tiempo = paso.groupby(df["estado"]).sum() / 60
nombres = {0: "Seguro", 1: "Aviso", 2: "Alerta", 3: "Peligro", 4: "Evacuación"}
tiempo.index = [nombres.get(int(i), str(i)) for i in tiempo.index]
print("\nMinutos en cada estado:")
print(tiempo.round(1))

print("\nNivel máximo:", round(df["nivel_limpio"].max(), 1), "cm a las",
      df.loc[df["nivel_limpio"].idxmax(), "created_at"].strftime("%d/%m %H:%M"))
# Concordancia entre los dos sensores: el ultrasónico marca 5 cm o más y el sensor del piso marca agua
ultra_mojado = df["nivel_limpio"] >= 5
piso_mojado = df["agua_piso_pct"] >= 50
print("Concordancia entre sensores:", round((ultra_mojado == piso_mojado).mean() * 100, 1), "% de las lecturas")

## 5. Gráficos

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(df["created_at"], df["nivel_limpio"], color="#14365A", lw=1.5, label="Nivel del agua")
colores = ["#E0B000", "#E07B00", "#C62828", "#6A1B9A"]
etiquetas = ["Aviso 5 cm", "Alerta 15 cm", "Peligro 30 cm", "Evacuación 60 cm"]
for u, c, e in zip(UMBRALES_CM, colores, etiquetas):
    ax.axhline(u, color=c, ls="--", lw=1, label=e)
ax.set_title("Nivel del agua durante el evento")
ax.set_ylabel("Nivel (cm sobre el piso)")
ax.set_xlabel("Hora")
ax.legend(loc="upper left", fontsize=8)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
axs[0].scatter(df["agua_piso_pct"], df["nivel_limpio"], s=6, alpha=0.4, color="#1F7A8C")
axs[0].set_title("Agua en el piso contra nivel")
axs[0].set_xlabel("Agua en el piso (%)")
axs[0].set_ylabel("Nivel (cm)")

axs[1].plot(df["created_at"], df["velocidad_cm_min"], color="#C62828", lw=1)
axs[1].axhline(3, color="grey", ls="--", lw=1, label="Subida rápida (3 cm/min)")
axs[1].set_title("Velocidad de subida")
axs[1].set_ylabel("cm por minuto")
axs[1].legend(fontsize=8)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 6. Eventos de alerta

In [ ]:
df["en_alerta"] = df["estado"] >= 1
df["evento"] = (df["en_alerta"] != df["en_alerta"].shift()).cumsum()

eventos = (df[df["en_alerta"]]
           .groupby("evento")
           .agg(inicio=("created_at", "min"), fin=("created_at", "max"),
                pico_cm=("nivel_limpio", "max"), estado_max=("estado", "max")))
eventos["duracion_min"] = ((eventos["fin"] - eventos["inicio"]).dt.total_seconds() / 60).round(1)
eventos["respondio"] = df[df["en_alerta"]].groupby("evento")["respuesta"].max().gt(0)
eventos

## 7. Guardar el resultado

In [ ]:
df.to_csv("datos_limpios.csv", index=False)
eventos.to_csv("eventos.csv")
print("Archivos guardados: datos_limpios.csv y eventos.csv (están en el panel de archivos de Colab)")